In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
import os
from dotenv import load_dotenv

load_dotenv()  # reads GOOGLE_API_KEY and ENDPOINT_URL from .env

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
ENDPOINT_URL   = os.getenv("ENDPOINT_URL")

# Create Gemini LLM instance
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=GOOGLE_API_KEY,
    base_url=ENDPOINT_URL,
    additional_headers={
        "Authorization": f"Bearer {GOOGLE_API_KEY}",
        "api-key": GOOGLE_API_KEY,
    },
    temperature=1,
    max_tokens=1024,
)
# Create memory checkpoint
memory = MemorySaver()
print(" Gemini LLM initialized.")
print(" MemorySaver checkpoint created.")


 Gemini LLM initialized.
 MemorySaver checkpoint created.


In [2]:

def draft_email(recipient_list: str, subject: str, body: str) -> str:

    """Draft an email to a list of friends requesting their availability."""

    return f"""Email drafted:

To: {recipient_list}

Subject: {subject}

Body: {body}

[Ready to send]"""

def record_response(friend_name: str, availability: str) -> str:

    """Record a friend's availability response."""

    return f"Recorded {friend_name}'s response: {availability}"

def book_restaurant(name: str, date: str, time: str, party_size: int) -> str:

    """Book a restaurant table."""

    return f"Restaurant '{name}' booked for {party_size} people on {date} at {time}"

print("Tools defined: draft_email, record_response, book_restaurant")


Tools defined: draft_email, record_response, book_restaurant


In [3]:

system_prompt = """Role: Professional butler, organizing social events and dinners.

Your Process:

1. Understand the user's event details (who, when, where preferences).
2. Draft an email invitation to friends requesting their availability.
3. Once you receive availability confirmation, consolidate responses.
4. Draft a restaurant booking call script with confirmed details.
5. Book the restaurant with final details (date, time, party size).
6. Draft a final confirmation email to all friends with venue, date, and time.

Key Behaviors:
- Break complex tasks into distinct steps.
- Use tools to draft communications and confirmations.
- Always confirm details before booking.
- Maintain a professional and organized tone.

Constraints:
- Do NOT book without confirmation.
- Do NOT send emails without drafting first.
- Always show your work and explain each step."""

# Create ReAct agent with memory

agent = create_agent(
    model=llm,
    tools=[draft_email, record_response, book_restaurant],
    checkpointer=memory,
    system_prompt=system_prompt,
)

print(" Scheduling agent created with MemorySaver.")

 Scheduling agent created with MemorySaver.


In [4]:
thread_id = "butler_session_001"
config = {"configurable": {"thread_id": thread_id}}
print(f" Thread ID: {thread_id}")
print(" Configuration ready.")

 Thread ID: butler_session_001
 Configuration ready.


In [5]:
initial_request = {
    "messages": [
        {
            "role": "user",
            "content": "I want to organize a dinner with my 5 college friends. Can you help me set this up?"
        }
    ]

}

print("=" * 60)
print("TURN 1: Initial Request")
print("=" * 60)

print("User: I want to organize a dinner with my 5 college friends. Can you help me set this up?")

print()

result = agent.invoke(initial_request, config)
response = format_conversation(result)[0]["text"]
print("Agent:")
print(response)
print()

print(f"Checkpoint: {len(result['messages'])} messages saved in thread '{thread_id}'")


TURN 1: Initial Request
User: I want to organize a dinner with my 5 college friends. Can you help me set this up?



NameError: name 'format_conversation' is not defined

In [ ]:

second_message = {
    "messages": [
        {
            "role": "user",
            "content": "Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant."
        }
    ]

}

print("=" * 60)
print("TURN 2: User Confirms Availability")
print("=" * 60)

print("User: Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant.")

print()
result = agent.invoke(second_message, config)
response = format_conversation(result)[0]["text"]
print("Agent:")
print(response)
print()
print(f"Checkpoint: {len(result['messages'])} messages saved")
print("Agent has full context from Turn 1 (multi-turn memory working).")


TURN 2: User Confirms Availability
User: Great! They all confirmed for next Friday evening around 7 PM. My friends are: Alice, Bob, Carol, Dave, and Eve. Please book a table for 6 at an Italian restaurant.

